In [2]:
import pandas as pd
from db_queries import get_location_metadata
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from tqdm import tqdm
import numpy as np
from matplotlib.ticker import MaxNLocator
from pathlib import Path
import yaml
import xarray as xr

# Set paths ####################################################################
RAW_DATA_PATH = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/training_data/2026_09_23.01/data.parquet"
GBD_FILE = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/input/gbd_prevalence/gbd_mean_child_mortality_prevalence.parquet"
INFERENCE_FILE_10_01 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/results/2026_10_01.01/ssp245.parquet"  #
RESULTS_10_01 = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/models/2026_09_24.03/"  #

PLOT_OUTPUT = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/plots/"


df_raw = pd.read_parquet(RAW_DATA_PATH)

# Add location names
locs = get_location_metadata(location_set_id=39, release_id=16)

df_raw = df_raw.merge(
    locs[["location_id", "location_name", "level", "super_region_name"]],
    on="location_id",
    how="left",
)

df_raw.head()

,ihme_loc_id,geospatial_id,psu,strata,line_id,hh_id,lat,long,lbd_admin2_id,indv_id,...,age_48_m,age_60_m,age_group,lower_bound,upper_bound,location_id,consumption_median_RT,location_name,level,super_region_name
0,IND,19909,19909,19922,3,27,25.217504,82.598196,72400,467681_19909_27_3_2_2015_9_2020_1_64_1_NA_NA,...,0,0,age_1_m,0,1,163,4.617627,India,3,South Asia
1,GHA,198,198,5,1,28,5.645717,-0.152658,92667,218572_198_28_1_2_2015_6_2017_8_26_1_33_NA,...,0,0,age_1_m,0,1,207,3.398277,Ghana,3,Sub-Saharan Africa
2,ZWE,946,946,17,1,72,-17.877540,31.055500,90799,21163_946_72_1_1_1999_4_2005_8_76_1_NA_NA,...,0,0,age_1_m,0,1,198,3.949926,Zimbabwe,3,Sub-Saharan Africa
3,IND,260602,260602,52,2,46,21.275709,84.861936,72213,157050_260602_46_2_1_2012_6_2015_2_44_1_NA_NA,...,0,0,age_1_m,0,1,163,4.158805,India,3,South Asia
4,IND,51062,51062,10,1,7,25.584927,85.507996,71893,157050_51062_7_1_1_2014_7_2014_4_9_1_NA_NA,...,0,0,age_1_m,0,1,163,4.384727,India,3,South Asia


In [3]:
# Two subsets of the residual step's prevalence plots ########################
# Reproduces what make_results_plots does, in this kernel, so it needs no
# rra_climate_health import:
#   1. read sev_means.parquet, clip the shifted columns to [0, 1] as the
#      diagnostics do
#   2. population-weight the most-detailed locations up the FHS hierarchy
#   3. draw one page per location, same line styles as the published pdf
# Then write two pdfs: Global + the 7 super regions, and the level-3 countries.
SUB_RV = "2026_10_01.01"
SUB_MEASURE = "child_mortality"
SUB_ROOT = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition"
SUB_SEV = f"{SUB_ROOT}/{SUB_MEASURE}/results/{SUB_RV}/sev_means.parquet"
SUB_HIER = f"{SUB_ROOT}/input/fhs_hierarchy.parquet"
SUB_POP_FUT = "/mnt/share/forecasting/data/32/future/population/future_population_s130v41/population.nc"
SUB_POP_PAST = "/mnt/share/forecasting/data/16/past/population/20250603_etl_run_id_417/population.nc"
SUB_POP_SCENARIO = 130
SUB_REFERENCE = "ssp245"
SUB_ANCHOR = 2023  # constants.LAST_GBD_YEAR, where the shift is read off

# line styles copied from residual_diagnostics.PREVALENCE_STYLE_MAP / LABEL_MAP.
# adjusted_constant_climate is absent from this run's sev_means; every use of
# this list is guarded on the column existing, so it simply drops out.
SUB_STYLE = [
    ("gbd_value", "black", "-", "GBD Prevalence"),
    ("adjusted_constant_climate", "lightgreen", "-", "Shifted constant climate"),
    ("adjusted_ssp585", "purple", "-", "Shifted SSP 5-8.5"),
    ("adjusted_ssp126", "green", "-", "Shifted SSP 1-2.6"),
    ("model_value", "blue", ":", "Original model"),
    ("residual_model_value", "blue", "--", "Residual model"),
    ("adjusted_model_value", "blue", "-", "Shifted model"),
]
# child_mortality reports a single group, 1 = under 5
SUB_AGE_LABEL = {1: "Under 5 (age_group_id 1)", 42: "Neonatal (age_group_id 42)"}

sub_res = pd.read_parquet(SUB_SEV)
sub_hier = pd.read_parquet(SUB_HIER)
_ages = sorted(sub_res.index.get_level_values("age_group_id").unique())
_sexes = sorted(sub_res.index.get_level_values("sex_id").unique())
_years = sorted(sub_res.index.get_level_values("year_id").unique())

# the clipping make_results_plots applies before aggregating
for _c in [c for c in sub_res.columns if c.startswith("adjusted_")]:
    sub_res[_c] = sub_res[_c].clip(0, 1)


# Reported age groups that the population files store only as their parts,
# mirroring constants.AGE_GROUP_AGGREGATES. child_mortality's 1 is all of
# under-5, so the population has to be summed over its six components first.
SUB_AGE_AGGREGATES = {
    4: [388, 389],
    5: [238, 34],
    42: [2, 3],
    1: [2, 3, 388, 389, 238, 34],
}


def sub_load_population():
    """Past + future population means, on the sev_means demographic grid."""
    detailed = sorted({a for age in _ages for a in SUB_AGE_AGGREGATES.get(age, [age])})
    fut = (
        xr.open_dataset(SUB_POP_FUT)
        .sel(age_group_id=detailed, sex_id=_sexes, scenario=SUB_POP_SCENARIO)
        .to_dataframe()
        .reset_index()
    )
    past = (
        xr.open_dataset(SUB_POP_PAST)
        .sel(age_group_id=detailed, sex_id=_sexes)
        .to_dataframe()
        .reset_index()
    )
    out = []
    for df in (past, fut):
        val = "population" if "population" in df.columns else "draws"
        keys = ["location_id", "year_id", "age_group_id", "sex_id"]
        df = (
            df.groupby(keys, as_index=False)[val]
            .mean()
            .rename(columns={val: "population"})
        )
        # roll the detailed groups back up into the reported ones
        parts = []
        for age in _ages:
            members = SUB_AGE_AGGREGATES.get(age, [age])
            sub = df[df.age_group_id.isin(members)].copy()
            sub["age_group_id"] = age
            parts.append(sub.groupby(keys, as_index=False)["population"].sum())
        out.append(pd.concat(parts))
    pop = pd.concat(out)
    # past and future overlap; the forecast file wins, matching the pipeline
    pop = pop.drop_duplicates(
        ["location_id", "year_id", "age_group_id", "sex_id"], keep="last"
    )
    return pop[pop.year_id.isin(_years)]


sub_pop = sub_load_population()

# FHS forecast under-5 mortality, for reference against the pipeline's own
# lines. year5q0.nc is the under-5 analogue of the day28q0 series used for
# neonatal, and is stored at age_group_id 1, matching sev_means exactly. These
# files already carry every level of the hierarchy - Global, super regions,
# regions, countries and subnationals - so they merge straight on without any
# aggregation of their own. Values are plain probabilities
# (rate_denominator = 1), the same scale as gbd_value, so they need no rescale.
SUB_FHS = {
    "fhs_with_cuts": (
        "/mnt/share/forecasting/data/32/future/death/"
        "20260717_q0_reference_s130v108_noreagg_nq0",
        "#d1495b",
        "-",
        "FHS forecasts with ODA cuts",
    ),
    "fhs_no_cuts": (
        "/mnt/share/forecasting/data/32/future/death/"
        "20260716_q0_reference_s130v98_nq0",
        "#d1495b",
        "--",
        "FHS forecasts without ODA cuts",
    ),
}


def sub_load_fhs():
    """Draw-averaged year5q0 for both ODA scenarios, keyed like sub_agg."""
    out = None
    for col, (path, *_rest) in SUB_FHS.items():
        da = xr.open_dataarray(f"{path}/year5q0.nc")
        df = (
            da.sel(scenario=SUB_POP_SCENARIO, sex_id=_sexes)
            .mean(dim="draw")
            .squeeze()
            .to_dataframe()
            .reset_index()
            .rename(columns={"value": col})
        )
        keys = ["location_id", "year_id", "sex_id", "age_group_id"]
        df = df[keys + [col]]
        out = df if out is None else out.merge(df, on=keys, how="outer")
    return out


sub_fhs = sub_load_fhs()
print(
    f"FHS overlay: {sub_fhs.location_id.nunique()} locations, "
    f"{int(sub_fhs.year_id.min())}-{int(sub_fhs.year_id.max())}, "
    f"age {sorted(sub_fhs.age_group_id.unique())}"
)

# population-weight every most-detailed location up its path_to_top_parent
_detailed = sub_hier[sub_hier["most_detailed"] == 1][
    ["location_id", "path_to_top_parent"]
]
_vals = [c for c, *_ in SUB_STYLE if c in sub_res.columns]
_flat = (
    sub_res.reset_index()
    .merge(_detailed, on="location_id", how="inner")
    .merge(
        sub_pop, on=["location_id", "year_id", "age_group_id", "sex_id"], how="inner"
    )
)
_flat["agg_id"] = _flat["path_to_top_parent"].str.split(",")
_exp = _flat.explode("agg_id")
_exp["agg_id"] = _exp["agg_id"].astype(int)
for _c in _vals:
    _exp[_c] = _exp[_c] * _exp["population"]
_keys = ["agg_id", "year_id", "age_group_id", "sex_id"]
sub_agg = _exp.groupby(_keys, as_index=False)[_vals + ["population"]].sum(min_count=1)
for _c in _vals:
    sub_agg[_c] = sub_agg[_c] / sub_agg["population"]
sub_agg = sub_agg.rename(columns={"agg_id": "location_id"})

_lv = sub_hier.set_index("location_id")["level"]
_nm = sub_hier.set_index("location_id")["location_name"]
print(
    f"aggregated to {sub_agg.location_id.nunique()} locations "
    f"from {_flat.location_id.nunique()} most-detailed inputs; "
    f"levels {sorted(set(_lv.reindex(sub_agg.location_id.unique()).dropna().astype(int)))}"
)


def sub_make_pdf(loc_ids, out_path, title_note):
    """One page per location: a sex-by-age grid, styled like the published pdf."""
    with PdfPages(out_path) as pdf:
        for loc_id in loc_ids:
            fig, axes = plt.subplots(
                len(_sexes),
                len(_ages),
                figsize=(14, 8),
                sharex=True,
                sharey=True,
                squeeze=False,
            )
            handles, labels = [], []
            for j, sex_id in enumerate(_sexes):
                for i, age_id in enumerate(_ages):
                    ax = axes[j, i]
                    blk = sub_agg[
                        (sub_agg.location_id == loc_id)
                        & (sub_agg.sex_id == sex_id)
                        & (sub_agg.age_group_id == age_id)
                    ].sort_values("year_id")
                    for col, colour, ls, lab in SUB_STYLE:
                        if col in blk.columns:
                            ax.plot(
                                blk.year_id, blk[col], color=colour, ls=ls, label=lab
                            )
                    # FHS reference forecasts; 2024-2050 only, so these appear
                    # as a segment rather than spanning the panel
                    f = sub_fhs[
                        (sub_fhs.location_id == loc_id)
                        & (sub_fhs.sex_id == sex_id)
                        & (sub_fhs.age_group_id == age_id)
                    ].sort_values("year_id")
                    for col, (_pth, colour, ls, lab) in SUB_FHS.items():
                        if col in f.columns and f[col].notna().any():
                            ax.plot(
                                f.year_id,
                                f[col],
                                color=colour,
                                ls=ls,
                                lw=1.8,
                                zorder=5,
                                label=lab,
                            )
                    # the anchor year: the shifted model meets GBD here by
                    # construction, and GBD itself stops here
                    ax.axvline(
                        SUB_ANCHOR,
                        color="0.45",
                        lw=1.1,
                        ls="-.",
                        zorder=1,
                        label=f"anchor year {SUB_ANCHOR}",
                    )
                    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)
                    if j == 0:
                        ax.set_title(SUB_AGE_LABEL.get(age_id, f"Age group {age_id}"))
                    ax.set_ylabel("Male" if sex_id == 1 else "Female")
                    if i > 0:
                        ax.set_ylabel("")
                        ax.tick_params(labelleft=False)
                    if j == len(_sexes) - 1:
                        ax.set_xlabel("Year")
                    if not handles:
                        handles, labels = ax.get_legend_handles_labels()
            fig.suptitle(
                f"{SUB_MEASURE.replace('_', ' ').capitalize()} Prevalence - "
                f"Location {loc_id}: {_nm.get(loc_id)}",
                fontsize=16,
            )
            fig.legend(
                handles, labels, loc="lower center", ncol=3, bbox_to_anchor=(0.5, 0)
            )
            fig.tight_layout(rect=[0, 0.05, 1, 1])
            pdf.savefig(fig)
            plt.close(fig)
    print(f"Saved {out_path} ({len(loc_ids)} pages, {title_note})")


_ordered = [l for l in sub_agg.location_id.unique() if l in _lv.index]
SUB_SETS = {
    "global_superregions": (
        [l for l in _ordered if _lv[l] in (0, 1)],
        "Global and the super regions",
    ),
    "countries": ([l for l in _ordered if _lv[l] == 3], "level-3 countries"),
}
for _tag, (_locs, _note) in SUB_SETS.items():
    sub_make_pdf(
        _locs,
        f"{PLOT_OUTPUT}{SUB_MEASURE}_prevalence_plots_{_tag}.pdf",
        _note,
    )

FHS overlay: 538 locations, 2024-2050, age [1]
aggregated to 513 locations from 473 most-detailed inputs; levels [0, 1, 2, 3, 4]
Saved /mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/plots/child_mortality_prevalence_plots_global_superregions.pdf (8 pages, Global and the super regions)
Saved /mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/plots/child_mortality_prevalence_plots_countries.pdf (204 pages, level-3 countries)


In [ ]:
# Interactive explorer for the residual-step prevalence results ##############
# Writes one self-contained HTML file to PLOT_OUTPUT: the same series the pdfs
# draw, but searchable by location and inspectable point by point. No CDN, no
# build step - the data is embedded and the chart is hand-rolled SVG, so it
# opens off the shared drive with nothing else available.
#
# Depends on sub_agg / sub_fhs / sub_hier from the cell above.
import html
import json

ART_PATH = f"{PLOT_OUTPUT}{SUB_MEASURE}_prevalence_explorer.html"
ART_ANCHOR = SUB_ANCHOR

# Series encoding. Colours are the dataviz reference palette, validated with
# the all-pairs gate in BOTH modes (worst CVD dE 9.2 light / 9.4 dark, worst
# normal-vision dE 24.0 / 20.9). Only THREE categorical hues are used: four or
# more cannot clear the all-pairs floors, and every line here shares one panel,
# so every pair really is on screen together.
#   blue   = our model, its three stages separated by dash pattern, not hue
#   orange = SSP 5-8.5 (high emissions), aqua = SSP 1-2.6 (low emissions)
#   ink / muted grey = external references (GBD, FHS); achromatic on purpose,
#                      so they read as context rather than competing for a hue
# Trailing int is paint order, low to high: the shifted model is the headline
# series, so it is drawn last and sits on top of everything it crosses.
ART_SERIES = [
    ("gbd_value", "GBD prevalence", "ink", "none", 2.5, "Reference", 5),
    ("model_value", "Original model", "blue", "2 3", 2, "Model", 1),
    ("residual_model_value", "Residual model", "blue", "7 4", 2, "Model", 2),
    ("adjusted_model_value", "Shifted model", "blue", "none", 2.5, "Model", 9),
    ("adjusted_ssp126", "Shifted SSP 1-2.6", "aqua", "none", 2, "Scenario", 4),
    ("adjusted_ssp585", "Shifted SSP 5-8.5", "orange", "none", 2, "Scenario", 3),
    ("fhs_no_cuts", "FHS, no ODA cuts", "muted", "7 4", 2, "Reference", 6),
    ("fhs_with_cuts", "FHS, with ODA cuts", "muted", "none", 2, "Reference", 7),
]

# ---- assemble the payload ---------------------------------------------------
_art_years = sorted(sub_agg.year_id.unique())
_art_yidx = {y: i for i, y in enumerate(_art_years)}
_art_cols = [c for c, *_ in ART_SERIES if c in sub_agg.columns]
_art_fhs_cols = [c for c in ("fhs_no_cuts", "fhs_with_cuts") if c in sub_fhs.columns]

# Sexes stay separate. A combined line is deliberately not offered: the FHS
# series carry no population, so theirs could only be an unweighted mean of two
# differently sized groups, which would not be comparable with the
# population-weighted model lines beside it.
_art_long = sub_agg
_art_fhs = sub_fhs

_art_meta = sub_hier.set_index("location_id")
_LV_NAME = {0: "Global", 1: "Super region", 2: "Region", 3: "Country",
            4: "Subnational", 5: "Subnational"}


def _art_round(v):
    """1e-6 of a probability - 10x finer than the 3-decimal percent on display."""
    return None if v is None or pd.isna(v) else round(float(v), 6)


_art_locs = {}
for _lid, _g in _art_long.groupby("location_id"):
    if _lid not in _art_meta.index:
        continue
    _row = _art_meta.loc[_lid]
    _by_sex = {}
    for _sex, _gs in _g.groupby("sex_id"):
        _gs = _gs.set_index("year_id")
        _f = _art_fhs[(_art_fhs.location_id == _lid) & (_art_fhs.sex_id == _sex)]
        _f = _f.set_index("year_id")
        _ser = {}
        for _c in _art_cols:
            _v = _gs[_c].reindex(_art_years)
            _ser[_c] = [_art_round(x) for x in _v]
        for _c in _art_fhs_cols:
            _v = _f[_c].reindex(_art_years) if len(_f) else None
            _ser[_c] = (
                [None] * len(_art_years) if _v is None
                else [_art_round(x) for x in _v]
            )
        _by_sex[str(int(_sex))] = _ser
    _art_locs[str(int(_lid))] = {
        "n": str(_row["location_name"]),
        "lv": int(_row["level"]),
        "lvn": _LV_NAME.get(int(_row["level"]), "Other"),
        "sr": (None if pd.isna(_row.get("super_region_name"))
               else str(_row.get("super_region_name"))),
        "s": _by_sex,
    }

# ---- summary statistics shown under the chart -------------------------------
# Country-level, population-weighted across sexes, so these are one number per
# country rather than one per sex. Cumulative differences are summed over years
# and carry units of percentage-point-years (pp-yr): a country 1 pp above GBD
# for 10 years scores 10.
_sum_keys = ["location_id", "year_id"]
_sw = sub_agg.copy()
_sum_cols = [c for c in ("gbd_value", "adjusted_model_value",
                         "adjusted_ssp126", "adjusted_ssp585")
             if c in _sw.columns]
for _c in _sum_cols:
    _sw[_c] = _sw[_c] * _sw["population"]
_sum = _sw.groupby(_sum_keys, as_index=False)[_sum_cols + ["population"]].sum(min_count=1)
for _c in _sum_cols:
    _sum[_c] = _sum[_c] / _sum["population"]
_sum["level"] = _sum.location_id.map(sub_hier.set_index("location_id")["level"])
_sum = _sum[_sum.level == 3]

# 1. distance from GBD, over the years GBD actually covers
_g = _sum[_sum.gbd_value.notna()].copy()
_g["d"] = 100 * (_g.adjusted_model_value - _g.gbd_value)
_gbd_rank = (
    _g.groupby("location_id")
    .agg(cum=("d", "sum"), n=("d", "size"))
    .assign(mean=lambda d: d.cum / d.n)
    .reindex(columns=["cum", "mean", "n"])
    .sort_values("cum", key=lambda c: c.abs(), ascending=False)
    .head(3)
)
_gbd_years = (int(_g.year_id.min()), int(_g.year_id.max()))

# 2. spread between the two climate scenarios, over the full horizon
_s = _sum.copy()
_s["d"] = 100 * (_s.adjusted_ssp585 - _s.adjusted_ssp126)
_scen_rank = (
    _s.groupby("location_id")
    .agg(cum=("d", "sum"), n=("d", "size"))
    .assign(mean=lambda d: d.cum / d.n)
    .sort_values("cum", key=lambda c: c.abs(), ascending=False)
    .head(3)
)
_scen_years = (int(_s.year_id.min()), int(_s.year_id.max()))

# 3. the residual model's fitted SDI slope per super region.
# Not stored anywhere by the pipeline, so it is recovered from the output: the
# model is intercept(location_age_sex) + slope(super_region) * sdi, so within a
# single location/age/sex an OLS of the predicted residual on SDI returns that
# region's slope exactly. Read against the SDI files this results version was
# FITTED with - rra-climate-health_2 has since been repointed at the round-38
# release, which would give the wrong slope here.
SDI_PAST_FITTED = "/mnt/share/forecasting/data/32/past/sdi/past_sdi_s130v89/sdi.nc"
SDI_FUTURE_FITTED = "/mnt/share/forecasting/data/32/future/sdi/future_sdi_s130v89/sdi.nc"
SDI_FITTED_SCENARIO = 130


def _art_load_sdi():
    _p = xr.open_dataset(SDI_PAST_FITTED).to_dataframe().reset_index()
    _p = _p.groupby(["location_id", "year_id"], as_index=False)["draws"].mean()
    _f = xr.open_dataset(SDI_FUTURE_FITTED).to_dataframe().reset_index()
    _f = _f[_f.scenario == SDI_FITTED_SCENARIO]
    _f = _f.groupby(["location_id", "year_id"], as_index=False)["draws"].mean()
    return (pd.concat([_p, _f]).rename(columns={"draws": "sdi"})
            .drop_duplicates(["location_id", "year_id"]))


_sdi = _art_load_sdi()
_res = sub_res.reset_index()
_res["pr"] = _res.residual_model_value - _res.model_value
_res = _res.merge(_sdi, on=["location_id", "year_id"], how="inner")
_res["srn"] = _res.location_id.map(sub_hier.set_index("location_id")["super_region_name"])
_slope_rows = []
for (_lid, _a, _x), _gg in _res.groupby(["location_id", "age_group_id", "sex_id"]):
    if _gg.sdi.nunique() < 3 or pd.isna(_gg.srn.iloc[0]):
        continue
    _slope_rows.append((_gg.srn.iloc[0], float(np.polyfit(_gg.sdi, _gg.pr, 1)[0])))
# Median, not mean: the recovery is approximate. The predicted residual is not
# perfectly linear in this SDI series (it departs by up to ~2e-4), so per-location
# slopes within one super region scatter by up to ~2e-3 instead of agreeing to
# machine precision. Reported to 3 dp, which is inside that scatter. The clipped
# regions are exact: every location in them returns 0.
_slopes = (
    pd.DataFrame(_slope_rows, columns=["srn", "slope"])
    .groupby("srn")["slope"]
    .agg(["median", "std", "size"])
    .sort_values("median")
)
print(
    f"summary: GBD window {_gbd_years[0]}-{_gbd_years[1]}, "
    f"scenario window {_scen_years[0]}-{_scen_years[1]}, "
    f"{len(_slopes)} super-region slopes "
    f"(max within-region spread {_slopes['std'].max():.1e})"
)
print(_slopes.to_string(float_format=lambda v: f"{v:10.4f}"))

# GBD vs the FHS no-ODA-cuts forecast. FHS carries no population of its own, so
# it is collapsed across sexes with the model's population weights for the same
# location/year/sex. NOTE the two series overlap in ONE year only: GBD ends at
# 2024 and FHS begins at 2024, so this is that year's gap, not a sum over time.
_fh = sub_fhs.merge(
    sub_agg[["location_id", "year_id", "sex_id", "age_group_id", "population"]],
    on=["location_id", "year_id", "sex_id", "age_group_id"], how="inner",
)
_fh["w"] = _fh["fhs_no_cuts"] * _fh["population"]
_fh = _fh.groupby(["location_id", "year_id"], as_index=False)[["w", "population"]].sum(
    min_count=1
)
_fh["fhs_no_cuts"] = _fh["w"] / _fh["population"]
_gf = _sum[_sum.gbd_value.notna()].merge(
    _fh[["location_id", "year_id", "fhs_no_cuts"]], on=["location_id", "year_id"],
    how="inner",
)
_gf["d"] = 100 * (_gf.gbd_value - _gf.fhs_no_cuts)
_fhs_rank = (
    _gf.groupby("location_id")
    .agg(cum=("d", "sum"), n=("d", "size"))
    .assign(mean=lambda d: d.cum / d.n)
    .sort_values("cum", key=lambda c: c.abs(), ascending=False)
    .head(3)
)
_fhs_years = sorted(_gf.year_id.unique())
print(
    f"GBD vs FHS overlap: {len(_fhs_years)} year(s) "
    f"({_fhs_years[0]}-{_fhs_years[-1]}) across {_gf.location_id.nunique()} countries"
)

_nm_of = sub_hier.set_index("location_id")["location_name"]
ART_SUMMARY = {
    "gbdWindow": list(_gbd_years),
    "scenWindow": list(_scen_years),
    "gbd": [
        {"id": int(i), "n": str(_nm_of.get(i)), "cum": round(float(r.cum), 2),
         "mean": round(float(r["mean"]), 3)}
        for i, r in _gbd_rank.iterrows()
    ],
    "fhsWindow": [int(_fhs_years[0]), int(_fhs_years[-1])],
    "fhs": [
        {"id": int(i), "n": str(_nm_of.get(i)), "cum": round(float(r.cum), 2),
         "mean": round(float(r["mean"]), 3)}
        for i, r in _fhs_rank.iterrows()
    ],
    "scen": [
        {"id": int(i), "n": str(_nm_of.get(i)), "cum": round(float(r.cum), 2),
         "mean": round(float(r["mean"]), 3)}
        for i, r in _scen_rank.iterrows()
    ],
    "slopes": [
        {"sr": str(i), "slope": round(float(r["median"]), 3),
         "sd": round(float(r["std"]), 4)}
        for i, r in _slopes.iterrows()
    ],
}

ART_PAYLOAD = {
    "measure": SUB_MEASURE.replace("_", " "),
    "resultsVersion": SUB_RV,
    "anchor": ART_ANCHOR,
    "years": [int(y) for y in _art_years],
    "series": [
        {"k": k, "l": lab, "c": col, "d": dash, "w": w, "g": grp, "z": z}
        for k, lab, col, dash, w, grp, z in ART_SERIES
        if k in _art_cols + _art_fhs_cols
    ],
    "locs": _art_locs,
    "summary": ART_SUMMARY,
}
print(
    f"explorer payload: {len(ART_PAYLOAD['locs'])} locations, "
    f"{len(ART_PAYLOAD['years'])} years, {len(ART_PAYLOAD['series'])} series"
)
ART_HTML = r"""<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>__TITLE__</title>
<style>
:root{
  color-scheme:light;
  --plane:#f9f9f7; --surface:#fcfcfb; --ink:#0b0b0b; --ink2:#52514e; --muted:#898781;
  --grid:#e1e0d9; --axis:#c3c2b7; --ring:rgba(11,11,11,.10);
  --blue:#2a78d6; --orange:#eb6834; --aqua:#1baf7a; --neutral:#898781; --inkline:#0b0b0b;
  --accent:#2a78d6;
}
@media (prefers-color-scheme:dark){ :root:where(:not([data-theme="light"])){
  color-scheme:dark;
  --plane:#0d0d0d; --surface:#1a1a19; --ink:#fff; --ink2:#c3c2b7; --muted:#898781;
  --grid:#2c2c2a; --axis:#383835; --ring:rgba(255,255,255,.10);
  --blue:#3987e5; --orange:#d95926; --aqua:#199e70; --neutral:#898781; --inkline:#fff;
  --accent:#3987e5;
}}
:root[data-theme="dark"]{
  color-scheme:dark;
  --plane:#0d0d0d; --surface:#1a1a19; --ink:#fff; --ink2:#c3c2b7; --muted:#898781;
  --grid:#2c2c2a; --axis:#383835; --ring:rgba(255,255,255,.10);
  --blue:#3987e5; --orange:#d95926; --aqua:#199e70; --neutral:#898781; --inkline:#fff;
  --accent:#3987e5;
}
*{box-sizing:border-box}
body{margin:0;background:var(--plane);color:var(--ink);
  font:14px/1.5 system-ui,-apple-system,"Segoe UI",sans-serif}
header{display:flex;align-items:baseline;gap:12px;flex-wrap:wrap;
  padding:14px 16px;border-bottom:1px solid var(--ring)}
h1{font-size:17px;margin:0;font-weight:650}
.sub{color:var(--ink2);font-size:12.5px}
.spacer{flex:1}
button,select,input{font:inherit;color:var(--ink);background:var(--surface);
  border:1px solid var(--ring);border-radius:7px;padding:6px 10px}
button{cursor:pointer}
button:hover{border-color:var(--muted)}
button[aria-pressed="true"]{background:var(--accent);border-color:var(--accent);color:#fff}
.wrap{display:grid;grid-template-columns:286px minmax(0,1fr);gap:16px;padding:16px;
  align-items:start}
.panel{background:var(--surface);border:1px solid var(--ring);border-radius:11px}
.wrap2{padding:0 16px 16px}
#side{padding:12px;position:sticky;top:16px;max-height:calc(100vh - 32px);
  display:flex;flex-direction:column}
#q{width:100%;margin-bottom:8px}
#list{flex:1;min-height:0;overflow:auto;margin:0;padding:0;list-style:none}
#list li{padding:6px 9px;border-radius:7px;cursor:pointer;display:flex;gap:8px;
  align-items:baseline}
#list li:hover{background:var(--plane)}
#list li[aria-selected="true"]{background:var(--accent);color:#fff}
#list li[aria-selected="true"] .lv{color:#fff;opacity:.85}
.lv{color:var(--muted);font-size:11px;margin-left:auto;white-space:nowrap}
.grp{padding:9px 9px 3px;font-size:11px;letter-spacing:.06em;text-transform:uppercase;
  color:var(--muted);font-weight:650}
#main{padding:14px 16px 18px}
.row{display:flex;gap:16px;align-items:center;flex-wrap:wrap;margin-bottom:10px}
.ctl{display:flex;gap:6px;align-items:center}
.ctl label{color:var(--ink2);font-size:12.5px}
h2{font-size:16px;margin:0 0 2px}
.meta{color:var(--ink2);font-size:12.5px;margin-bottom:10px}
#legend{display:flex;flex-wrap:wrap;gap:6px 14px;margin:10px 0 2px}
.lg{display:flex;gap:7px;align-items:center;cursor:pointer;border:0;background:none;
  padding:3px 5px;border-radius:6px;color:var(--ink2);font-size:12.5px}
.lg:hover{background:var(--plane)}
.lg[aria-pressed="false"]{opacity:.38}
.lg svg{flex:none}
.grpname{font-size:11px;color:var(--muted);align-self:center;margin-right:-6px}
figure{margin:0}
#chart{width:100%;height:auto;display:block;touch-action:none}
#tip{position:fixed;pointer-events:none;z-index:9;background:var(--surface);
  border:1px solid var(--ring);border-radius:9px;padding:9px 11px;font-size:12.5px;
  box-shadow:0 6px 22px rgba(0,0,0,.14);opacity:0;transition:opacity .08s;max-width:290px}
#tip b{font-weight:650}
#tip table{border-collapse:collapse;margin-top:5px}
#tip td{padding:1px 0}
#tip td.v{text-align:right;padding-left:12px;font-variant-numeric:tabular-nums}
#tbl{width:100%;border-collapse:collapse;font-size:12.5px;
  font-variant-numeric:tabular-nums}
#tbl th,#tbl td{padding:4px 8px;text-align:right;border-bottom:1px solid var(--grid)}
#tbl th:first-child,#tbl td:first-child{text-align:left}
#tbl thead th{position:sticky;top:0;background:var(--surface);color:var(--ink2);
  font-weight:600}
#tblwrap{max-height:62vh;overflow:auto;margin-top:10px}
.hidden{display:none !important}
#sumgrid{display:grid;grid-template-columns:repeat(auto-fit,minmax(290px,1fr));gap:18px}
.sumcard h3{font-size:13px;margin:0 0 2px;font-weight:650}
.sumcard p{margin:0 0 7px;color:var(--muted);font-size:11.5px}
.sumtbl{width:100%;border-collapse:collapse;font-size:12.5px;
  font-variant-numeric:tabular-nums}
.sumtbl th,.sumtbl td{padding:4px 7px;text-align:right;border-bottom:1px solid var(--grid)}
.sumtbl th:first-child,.sumtbl td:first-child{text-align:left}
.sumtbl thead th{color:var(--ink2);font-weight:600}
.sumtbl tbody tr{cursor:pointer}
.sumtbl tbody tr:hover{background:var(--plane)}
.sumtbl .neg{color:var(--ink2)}
footer{color:var(--muted);font-size:11.5px;padding:0 16px 22px;line-height:1.6}
kbd{border:1px solid var(--ring);border-bottom-width:2px;border-radius:4px;
  padding:0 4px;font:inherit;font-size:11px}
@media (max-width:860px){.wrap{grid-template-columns:1fr}#side{position:static}
  #list{max-height:230px}}
</style></head><body>
<header>
  <h1>__TITLE__</h1>
  <span class="sub">results __RV__ &middot; shifted to GBD at __ANCHOR__</span>
  <span class="spacer"></span>
  <button id="view" aria-pressed="false">Table view</button>
  <button id="csv">Download CSV</button>
  <button id="theme">Theme</button>
</header>
<div class="wrap">
  <nav class="panel" id="side">
    <input id="q" type="search" placeholder="Search locations&hellip;  ( / )"
           autocomplete="off" aria-label="Search locations">
    <ul id="list" role="listbox" aria-label="Locations"></ul>
  </nav>
  <section class="panel" id="main">
    <h2 id="locname">&nbsp;</h2>
    <div class="meta" id="locmeta">&nbsp;</div>
    <div class="row">
      <div class="ctl"><label for="sex">Sex</label>
        <select id="sex"><option value="1">Male</option>
          <option value="2">Female</option></select></div>
      <div class="ctl"><label for="scale">Y axis</label>
        <select id="scale"><option value="lin">Linear</option>
          <option value="log">Log</option></select></div>
      <div class="ctl"><label for="xmax">Through</label>
        <select id="xmax"><option value="2100">2100</option>
          <option value="2050">2050</option><option value="2035">2035</option></select></div>
    </div>
    <figure id="figwrap">
      <svg id="chart" role="img" aria-label="Prevalence over time"></svg>
      <div id="legend"></div>
    </figure>
    <div id="tblwrap" class="hidden"><table id="tbl"></table></div>
  </section>
</div>
<div class="wrap2">
  <section class="panel" id="summary">
    <h2>Summary statistics</h2>
    <div class="meta">Country level (level 3), population-weighted across sexes.
      Cumulative differences are summed over years, in percentage-point-years
      (pp-yr): 1 pp of separation sustained for 10 years scores 10.</div>
    <div id="sumgrid"></div>
  </section>
</div>
<div id="tip" role="tooltip"></div>
<footer>
  <div>Lines are the residual step's own output, population-weighted up the FHS
  hierarchy. The dashed vertical marks __ANCHOR__, the last GBD year and the point
  the shifted model is pinned to; GBD itself stops there. FHS reference forecasts
  cover 2024&ndash;2050 only.</div>
  <div>Sexes are shown separately: the FHS series carry no population, so a
  combined line could only be an unweighted mean and would not be comparable with
  the population-weighted model lines. On a log axis, values at or below zero
  cannot be drawn &mdash; the residual model runs negative in many locations, so
  use the linear axis to see it.</div>
  <div><kbd>/</kbd> search &middot; <kbd>&uarr;</kbd><kbd>&darr;</kbd> move
  &middot; <kbd>Enter</kbd> select &middot; click a legend entry to hide a series.</div>
</footer>
<script>
const D = __PAYLOAD__;
const S = D.series, YEARS = D.years;
const COLORVAR = {ink:'--inkline', blue:'--blue', orange:'--orange', aqua:'--aqua',
                  muted:'--neutral'};
const css = n => getComputedStyle(document.documentElement).getPropertyValue(n).trim();
const ids = Object.keys(D.locs);
const LVORDER = {0:0, 1:1, 2:2, 3:3, 4:4, 5:5};
ids.sort((a,b)=>{
  const A=D.locs[a], B=D.locs[b];
  return (LVORDER[A.lv]-LVORDER[B.lv]) || A.n.localeCompare(B.n);
});
let cur = ids.find(i=>D.locs[i].lv===0) || ids[0];
let hidden = new Set(), filtered = ids.slice(), hi = 0;

/* ---- location list ---- */
const listEl = document.getElementById('list');
function renderList(){
  const q = document.getElementById('q').value.trim().toLowerCase();
  filtered = ids.filter(i=>{
    const L = D.locs[i];
    return !q || L.n.toLowerCase().includes(q) ||
           (L.sr||'').toLowerCase().includes(q);
  });
  if(hi >= filtered.length) hi = Math.max(0, filtered.length-1);
  let out = '', last = null;
  filtered.forEach((i,k)=>{
    const L = D.locs[i];
    if(L.lvn !== last){ out += `<div class="grp">${L.lvn}</div>`; last = L.lvn; }
    out += `<li role="option" data-id="${i}" data-k="${k}"
      aria-selected="${i===cur}" tabindex="-1"><span>${esc(L.n)}</span>
      <span class="lv">${L.sr?esc(abbr(L.sr)):''}</span></li>`;
  });
  listEl.innerHTML = out || '<div class="grp">No match</div>';
}
const esc = s => s.replace(/[&<>"]/g, c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c]));
const abbr = s => s.replace('Sub-Saharan Africa','SSA')
  .replace('Central Europe, Eastern Europe, and Central Asia','C/E Europe & C Asia')
  .replace('Southeast Asia, East Asia, and Oceania','SE/E Asia & Oceania')
  .replace('Latin America and Caribbean','LatAm & Caribbean')
  .replace('North Africa and Middle East','N Africa & Middle East');

listEl.addEventListener('click', e=>{
  const li = e.target.closest('li'); if(!li) return;
  cur = li.dataset.id; hi = +li.dataset.k; location.hash = 'loc='+cur; draw();
});
const qEl = document.getElementById('q');
qEl.addEventListener('input', ()=>{ hi=0; renderList(); });
qEl.addEventListener('keydown', e=>{
  if(e.key==='ArrowDown'||e.key==='ArrowUp'){
    e.preventDefault();
    hi = Math.min(filtered.length-1, Math.max(0, hi + (e.key==='ArrowDown'?1:-1)));
    const li = listEl.querySelector(`li[data-k="${hi}"]`);
    if(li){ listEl.querySelectorAll('li').forEach(x=>x.setAttribute('aria-selected','false'));
            li.setAttribute('aria-selected','true'); li.scrollIntoView({block:'nearest'}); }
  } else if(e.key==='Enter' && filtered[hi]){
    cur = filtered[hi]; location.hash='loc='+cur; draw();
  }
});
document.addEventListener('keydown', e=>{
  if(e.key==='/' && document.activeElement!==qEl){ e.preventDefault(); qEl.focus(); qEl.select(); }
});

/* ---- chart ---- */
const svg = document.getElementById('chart'), tip = document.getElementById('tip');
const NS = 'http://www.w3.org/2000/svg';
const el = (t,a)=>{ const n=document.createElementNS(NS,t);
  for(const k in a) n.setAttribute(k,a[k]); return n; };
const fmt = v => v==null ? '—' : (100*v).toFixed(3)+'%';
let geom = null;

function activeSeries(){ return S.filter(s=>!hidden.has(s.k)); }
// legend and tooltip keep the declared order; only painting is re-sorted
function paintOrder(){ return activeSeries().slice().sort((a,b)=>a.z-b.z); }
function seriesData(){
  const sex = document.getElementById('sex').value;
  return (D.locs[cur].s[sex]) || {};
}
function draw(){
  const L = D.locs[cur], data = seriesData();
  document.getElementById('locname').textContent = L.n;
  document.getElementById('locmeta').textContent =
    [L.lvn, L.sr||null, 'location_id '+cur].filter(Boolean).join('  ·  ');
  listEl.querySelectorAll('li').forEach(li=>
    li.setAttribute('aria-selected', String(li.dataset.id===cur)));
  renderLegend(); drawChart(data); renderTable(data);
}
function drawChart(data){
  const W = svg.clientWidth || 900, H = Math.max(330, Math.min(520, W*0.46));
  const m = {t:14, r:16, b:34, l:58};
  svg.setAttribute('viewBox', `0 0 ${W} ${H}`);
  svg.setAttribute('height', H);
  svg.textContent = '';
  const xmax = +document.getElementById('xmax').value;
  const yrs = YEARS.filter(y=>y<=xmax);
  const logy = document.getElementById('scale').value === 'log';
  const act = activeSeries();
  let vals = [];
  act.forEach(s=>(data[s.k]||[]).forEach((v,i)=>{
    if(v!=null && YEARS[i]<=xmax && (!logy || v>0)) vals.push(v); }));
  if(!vals.length) vals = [0, 1e-3];
  // Linear: the domain is whatever the data spans, including negatives - the
  // residual model runs far below zero in many locations, and a floor of 0
  // would silently clip it. Log: non-positive values cannot be shown at all,
  // so they are dropped from `vals` above and the footnote says so.
  let lo, hiV = Math.max(...vals);
  if(logy){ lo = Math.min(...vals); if(hiV<=lo) hiV = lo*1.5+1e-9; lo*=0.85; hiV*=1.15; }
  else {
    lo = Math.min(0, Math.min(...vals));
    if(hiV<=lo) hiV = lo + 1e-9;
    const pad = (hiV-lo)*0.06; hiV += pad; if(lo<0) lo -= pad;
  }
  const x = y => m.l + (y-yrs[0])/(xmax-yrs[0] || 1)*(W-m.l-m.r);
  const Y = v => { if(v==null) return null;
    const f = logy ? (Math.log(Math.max(v,1e-12))-Math.log(lo))/(Math.log(hiV)-Math.log(lo))
                   : (v-lo)/(hiV-lo);
    return m.t + (1-f)*(H-m.t-m.b); };
  geom = {x, Y, m, W, H, yrs, data, lo, hiV};

  const ticks = logy ? logTicks(lo,hiV) : linTicks(lo,hiV,5);
  ticks.forEach(t=>{
    const yy = Y(t); if(yy==null||yy<m.t-1||yy>H-m.b+1) return;
    svg.appendChild(el('line',{x1:m.l,x2:W-m.r,y1:yy,y2:yy,stroke:css('--grid'),'stroke-width':1}));
    const lb = el('text',{x:m.l-8,y:yy+4,'text-anchor':'end','font-size':11,fill:css('--muted')});
    lb.textContent = (100*t).toFixed(t<0.01?2:(t<0.1?1:0))+'%'; svg.appendChild(lb);
  });
  const step = (xmax-yrs[0])>60?20:((xmax-yrs[0])>30?10:5);
  for(let yv=yrs[0]; yv<=xmax; yv+=step){
    const xx=x(yv);
    const lb = el('text',{x:xx,y:H-m.b+18,'text-anchor':'middle','font-size':11,fill:css('--muted')});
    lb.textContent = yv; svg.appendChild(lb);
  }
  svg.appendChild(el('line',{x1:m.l,x2:W-m.r,y1:H-m.b,y2:H-m.b,
    stroke:css('--axis'),'stroke-width':1}));
  if(!logy && lo < 0){
    const z = Y(0);
    svg.appendChild(el('line',{x1:m.l,x2:W-m.r,y1:z,y2:z,stroke:css('--axis'),
      'stroke-width':1.5}));
    const zl = el('text',{x:m.l+4,y:z-4,'font-size':11,fill:css('--muted')});
    zl.textContent='0'; svg.appendChild(zl);
  }
  if(D.anchor<=xmax){
    svg.appendChild(el('line',{x1:x(D.anchor),x2:x(D.anchor),y1:m.t,y2:H-m.b,
      stroke:css('--muted'),'stroke-width':1,'stroke-dasharray':'4 4'}));
    const a = el('text',{x:x(D.anchor)+5,y:m.t+12,'font-size':11,fill:css('--muted')});
    a.textContent = D.anchor; svg.appendChild(a);
  }
  paintOrder().forEach(s=>{
    const arr = data[s.k]||[]; let d='', pen=false;
    yrs.forEach((yv,i)=>{ const v=arr[i], yy=Y(v);
      if(v==null||yy==null||(logy&&v<=0)){ pen=false; return; }
      d += (pen?'L':'M') + x(yv).toFixed(1) + ' ' + yy.toFixed(1) + ' '; pen=true; });
    if(d) svg.appendChild(el('path',{d, fill:'none', stroke:css(COLORVAR[s.c]),
      'stroke-width':s.w, 'stroke-linecap':'round', 'stroke-linejoin':'round',
      ...(s.d!=='none'?{'stroke-dasharray':s.d}:{})}));
  });
  const cross = el('g',{id:'cross',visibility:'hidden'}); svg.appendChild(cross);
  cross.appendChild(el('line',{id:'cl',y1:m.t,y2:H-m.b,stroke:css('--muted'),'stroke-width':1}));
  svg.appendChild(el('rect',{x:m.l,y:m.t,width:Math.max(0,W-m.l-m.r),
    height:Math.max(0,H-m.t-m.b),fill:'transparent',id:'hit'}));
  const hit = svg.querySelector('#hit');
  hit.addEventListener('pointermove', onMove);
  hit.addEventListener('pointerleave', ()=>{ cross.setAttribute('visibility','hidden');
    tip.style.opacity=0; });
}
function linTicks(lo,hi,n){
  const raw=(hi-lo)/n, mag=Math.pow(10,Math.floor(Math.log10(raw)));
  const st=[1,2,2.5,5,10].map(m=>m*mag).find(s=>s>=raw)||mag*10;
  const out=[]; for(let v=Math.ceil(lo/st)*st; v<=hi; v+=st) out.push(v); return out;
}
function logTicks(lo,hi){
  const out=[]; for(let e=Math.floor(Math.log10(lo)); e<=Math.ceil(Math.log10(hi)); e++)
    for(const m of [1,2,5]){ const v=m*Math.pow(10,e); if(v>=lo&&v<=hi) out.push(v); }
  return out.length?out:[lo,hi];
}
function onMove(ev){
  if(!geom) return;
  const r = svg.getBoundingClientRect(), sc = geom.W / r.width;
  const px = (ev.clientX - r.left) * sc;
  let best=geom.yrs[0], bd=1e9;
  geom.yrs.forEach(y=>{ const d=Math.abs(geom.x(y)-px); if(d<bd){bd=d;best=y;} });
  const i = YEARS.indexOf(best);
  const cross = svg.querySelector('#cross'); cross.setAttribute('visibility','visible');
  const cl = cross.querySelector('#cl');
  cl.setAttribute('x1', geom.x(best)); cl.setAttribute('x2', geom.x(best));
  cross.querySelectorAll('circle').forEach(c=>c.remove());
  let rows='';
  activeSeries().forEach(s=>{
    const v=(geom.data[s.k]||[])[i]; if(v==null) return;
    const yy=geom.Y(v); if(yy==null) return;
    cross.appendChild(el('circle',{cx:geom.x(best),cy:yy,r:4,
      fill:css(COLORVAR[s.c]),stroke:css('--surface'),'stroke-width':2}));
    rows += `<tr><td><svg width="16" height="8"><line x1="0" y1="4" x2="16" y2="4"
      stroke="${css(COLORVAR[s.c])}" stroke-width="${s.w}"
      ${s.d!=='none'?`stroke-dasharray="${s.d}"`:''}/></svg> ${esc(s.l)}</td>
      <td class="v">${fmt(v)}</td></tr>`;
  });
  tip.innerHTML = `<b>${esc(D.locs[cur].n)} &middot; ${best}</b><table>${rows}</table>`;
  tip.style.opacity = 1;
  const tw = tip.offsetWidth, th = tip.offsetHeight;
  let lft = ev.clientX + 16, top = ev.clientY - th/2;
  if(lft + tw > innerWidth - 8) lft = ev.clientX - tw - 16;
  tip.style.left = Math.max(8,lft)+'px';
  tip.style.top = Math.min(Math.max(8,top), innerHeight-th-8)+'px';
}
/* ---- legend ---- */
function renderLegend(){
  const box = document.getElementById('legend'); box.textContent='';
  let g = null;
  S.forEach(s=>{
    if(s.g!==g){ g=s.g; const t=document.createElement('span');
      t.className='grpname'; t.textContent=g; box.appendChild(t); }
    const b=document.createElement('button');
    b.className='lg'; b.setAttribute('aria-pressed', String(!hidden.has(s.k)));
    b.innerHTML = `<svg width="22" height="10"><line x1="0" y1="5" x2="22" y2="5"
      stroke="${css(COLORVAR[s.c])}" stroke-width="${s.w}"
      ${s.d!=='none'?`stroke-dasharray="${s.d}"`:''} stroke-linecap="round"/></svg>
      <span>${esc(s.l)}</span>`;
    b.onclick = ()=>{ hidden.has(s.k)?hidden.delete(s.k):hidden.add(s.k); draw(); };
    box.appendChild(b);
  });
}
/* ---- table view ---- */
function renderTable(data){
  const act = activeSeries();
  let h = '<thead><tr><th>Year</th>' + act.map(s=>`<th>${esc(s.l)}</th>`).join('')
        + '</tr></thead><tbody>';
  const xmax = +document.getElementById('xmax').value;
  YEARS.forEach((y,i)=>{ if(y>xmax) return;
    h += `<tr><td>${y}</td>` + act.map(s=>`<td>${fmt((data[s.k]||[])[i])}</td>`).join('') + '</tr>';
  });
  document.getElementById('tbl').innerHTML = h + '</tbody>';
}
document.getElementById('view').onclick = e=>{
  const showTable = e.target.getAttribute('aria-pressed') !== 'true';
  e.target.setAttribute('aria-pressed', String(showTable));
  document.getElementById('figwrap').classList.toggle('hidden', showTable);
  document.getElementById('tblwrap').classList.toggle('hidden', !showTable);
};
document.getElementById('csv').onclick = ()=>{
  const data = seriesData(), act = activeSeries();
  let csv = 'location_id,location_name,sex_id,year_id,' + act.map(s=>s.k).join(',') + '\n';
  const sex = document.getElementById('sex').value;
  YEARS.forEach((y,i)=>{ csv += `${cur},"${D.locs[cur].n}",${sex},${y},`
    + act.map(s=>{ const v=(data[s.k]||[])[i]; return v==null?'':v; }).join(',') + '\n'; });
  const a=document.createElement('a');
  a.href = URL.createObjectURL(new Blob([csv],{type:'text/csv'}));
  a.download = `${D.measure.replace(/ /g,'_')}_${D.locs[cur].n.replace(/[^A-Za-z0-9]+/g,'_')}.csv`;
  a.click(); URL.revokeObjectURL(a.href);
};
document.getElementById('theme').onclick = ()=>{
  const now = document.documentElement.getAttribute('data-theme');
  const dark = now ? now==='dark'
    : matchMedia('(prefers-color-scheme: dark)').matches;
  document.documentElement.setAttribute('data-theme', dark?'light':'dark');
  draw(); renderSummary();
};
['sex','scale','xmax'].forEach(id=>document.getElementById(id).onchange = draw);
addEventListener('resize', ()=>{ if(geom) drawChart(seriesData()); });
/* ---- summary ---- */
function renderSummary(){
  const U = D.summary; if(!U) return;
  const rank = (rows, w) => `<table class="sumtbl"><thead><tr><th>Country</th>
    <th>Cumulative</th><th>Mean/yr</th></tr></thead><tbody>` + rows.map(r=>
    `<tr data-id="${r.id}" title="Show ${esc(r.n)}"><td>${esc(r.n)}</td>
     <td class="${r.cum<0?'neg':''}">${r.cum>0?'+':''}${r.cum.toFixed(1)} pp-yr</td>
     <td class="${r.mean<0?'neg':''}">${r.mean>0?'+':''}${r.mean.toFixed(2)} pp</td></tr>`
    ).join('') + '</tbody></table>';
  document.getElementById('sumgrid').innerHTML = `
    <div class="sumcard"><h3>Furthest from GBD</h3>
      <p>Shifted model minus GBD, ${U.gbdWindow[0]}&ndash;${U.gbdWindow[1]} (the
      years GBD covers). Ranked on absolute value.</p>${rank(U.gbd)}</div>
    <div class="sumcard"><h3>Widest scenario spread</h3>
      <p>Shifted SSP 5-8.5 minus Shifted SSP 1-2.6, ${U.scenWindow[0]}&ndash;${U.scenWindow[1]}.
      Ranked on absolute value.</p>${rank(U.scen)}</div>
    <div class="sumcard"><h3>Furthest from FHS</h3>
      <p>GBD minus the FHS no-ODA-cuts forecast. The two series meet in
      ${U.fhsWindow[0]===U.fhsWindow[1] ? U.fhsWindow[0]+' only - GBD ends there and FHS begins there'
        : U.fhsWindow[0]+'&ndash;'+U.fhsWindow[1]}, so this is that gap, not a
      sum over time. Ranked on absolute value.</p>${rank(U.fhs)}</div>
    <div class="sumcard"><h3>Residual model: SDI slope</h3>
      <table class="sumtbl"><thead><tr><th>Super region</th><th>Slope</th></tr></thead>
      <tbody>${U.slopes.map(r=>`<tr><td>${esc(r.sr)}</td>
        <td class="${r.slope<0?'neg':''}">${r.slope===0?'0 (clipped)':r.slope.toFixed(3)}</td>
        </tr>`).join('')}</tbody></table></div>`;
  document.querySelectorAll('#sumgrid tbody tr[data-id]').forEach(tr=>{
    tr.onclick = ()=>{ if(D.locs[tr.dataset.id]){ cur=tr.dataset.id;
      location.hash='loc='+cur; draw();
      document.getElementById('main').scrollIntoView({behavior:'smooth',block:'start'}); } };
  });
}
const hashId = (location.hash.match(/loc=(\d+)/)||[])[1];
if(hashId && D.locs[hashId]) cur = hashId;
renderList(); draw(); renderSummary();
</script></body></html>
"""

_art_title = f"{ART_PAYLOAD['measure'].title()} prevalence explorer"
_art_doc = (
    ART_HTML.replace("__TITLE__", html.escape(_art_title))
    .replace("__RV__", html.escape(SUB_RV))
    .replace("__ANCHOR__", str(ART_ANCHOR))
    # separators= drops the spaces json.dumps adds; </ is escaped so a value can
    # never close the <script> early
    .replace(
        "__PAYLOAD__",
        json.dumps(ART_PAYLOAD, separators=(",", ":")).replace("</", "<\\/"),
    )
)
Path(ART_PATH).write_text(_art_doc, encoding="utf-8")
print(f"Saved {ART_PATH} ({len(_art_doc) / 1e6:.1f} MB)")